# 第17章 网络压缩 (Network Compression)

> "GPT可能有万亿参数，但要在手机上跑AI，必须把模型'压扁'。"

## 1. 知识地图：章节结构与概览

```
网络压缩
├── 为什么需要网络压缩？
│   ├── 边缘设备（智能手表、自驾车传感器）资源受限
│   ├── 延迟问题（不能等云端往返）
│   └── 隐私问题（数据不离开设备）
├── 17.1 网络剪枝 (Network Pruning)
│   ├── 评估参数/神经元重要性
│   ├── 迭代剪枝 + 微调策略
│   ├── 权重剪枝 vs 神经元剪枝
│   │   ├── 权重剪枝：不规则网络，GPU难加速
│   │   └── 神经元剪枝：网络规整，PyTorch友好
│   ├── 为什么大网络比小网络好训练？
│   └── 彩票假说 (Lottery Ticket Hypothesis)
│       ├── 大网络 = 多张小彩票的组合
│       ├── "中奖" = 好初始化的子网络
│       ├── ICLR 2019 最佳论文
│       ├── 后续：正负号比数值更重要
│       └── 争议：不是在所有条件下都成立
├── 17.2 知识蒸馏 (Knowledge Distillation)
│   ├── 教师网络（大）→ 学生网络（小）
│   ├── 为什么有用？类间关系信息的传递
│   ├── 温度 T 的技巧
│   │   ├── y'_i = exp(y_i/T) / Σ_j exp(y_j/T)
│   │   ├── T=1 标准Softmax
│   │   ├── T>1 分布更平滑，揭示类间关系
│   │   └── T→∞ 均匀分布（无意义）
│   ├── 集成教师网络的知识蒸馏
│   └── 多层匹配：不止最后一层
├── 17.3 参数量化 (Parameter Quantization)
│   ├── 降低精度：32bit → 16bit → 8bit
│   ├── 权重聚类 + 查表
│   ├── 哈夫曼编码压缩
│   └── 二值网络 (Binary Network)
│       ├── 权重只有 +1 或 -1
│       ├── BinaryConnect, XNOR-Net
│       └── 意外好处：防止过拟合
├── 17.4 网络架构设计
│   ├── 标准卷积回顾：K×K×I×O 参数
│   ├── 深度可分离卷积
│   │   ├── 深度卷积：每个通道单独滤波
│   │   └── 点卷积(1×1)：通道间混合
│   ├── 参数量对比：(K²I + IO) / (K²IO) ≈ 1/K²
│   ├── 与低秩近似的关系
│   └── MobileNet 的成功
└── 17.5 动态计算 (Dynamic Computation)
    ├── 为什么需要？同一模型适配不同设备/电量
    ├── 动态深度：简单样本少走几层
    │   └── 每一层都接出分类器，综合所有输出
    └── 动态宽度：根据资源调整通道数
        └── Slimmable Neural Networks
```

## 2. 为什么需要网络压缩？

**三个核心原因：**

1. **延迟**：数据传到云端再返回，时间差不可接受。以自动驾驶传感器为例，必须即时响应
2. **隐私**：在智能手表上直接计算，不需要把个人数据传到云端
3. **资源受限**：边缘设备内存小、算力少，跑不动大模型

**五种技术可以组合使用**，获得乘性压缩效果：先改架构设计 → 做知识蒸馏 → 剪枝 → 量化 → 可选动态计算

## 3. 网络剪枝 (Network Pruning) 详解

### 3.1 剪枝框架

1. 先训练一个大的网络
2. 评估每个参数/神经元的重要性
3. 剪掉不重要的部分
4. 微调（fine-tune）恢复性能
5. 重复步骤2-4，逐步剪枝

**为什么不能一次剪太多？** 实验发现一次剪掉大量参数对网络的伤害太大，微调无法恢复。所以要逐步剪（每次10%左右），反复迭代。

### 3.2 权重剪枝 vs 神经元剪枝

| 维度 | 权重剪枝 | 神经元剪枝 |
|------|------|------|
| 剪枝单位 | 单个连接（权重） | 整颗神经元 |
| 网络形状 | 不规则（稀疏矩阵） | 规则（维度减小） |
| GPU加速 | 难（非规则稀疏矩阵乘法） | 易（标准矩阵乘法） |
| PyTorch实现 | 复杂 | 简单 |
| 实际加速 | 几乎无加速 | 真正加速 |

**重要事实：** 权重剪枝到95%稀疏，实际速度几乎没有加速甚至可能变慢，因为不规则稀疏矩阵无法利用GPU的矩阵乘法优化。所以实践中神经元剪枝更常用。

### 3.3 为什么大网络比小网络好训练？

**现象：** 先训练一个大网络再剪枝，得到的小网络效果比直接训练同大小的小网络好。

这引出了著名的**彩票假说**。

## 4. 彩票假说 (Lottery Ticket Hypothesis) 完整解析

### 4.1 类比解释

买彩票时，买越多张中奖概率越高。一个大网络可以看作**包含许多子网络的集合**，就像一次买了很多张彩票。训练大网络时，等于同时训练所有子网络。只要其中一个子网络能成功训练（"中奖"），整个大网络就成功了。

### 4.2 实验验证

1. 随机初始化一个大网络 → 训练 → 剪枝（保留"中奖"子网络）
2. 将这个子网络**重新随机初始化** → 重训练 → 训不起来！
3. 将子网络**用原大网络中对应的初始化参数**重训 → 训起来了！

结论：不是子网络结构好，而是**那组初始化参数"中奖"了**。

### 4.3 深入：正负号比数值更重要

"Deconstructing Lottery Tickets" 论文发现：
- 把"中奖"子网络的参数全部替换为 ±α（保留正负号，统一幅度）
- 效果与原始初始化参数差不多！
- 正负号才是初始化成功的关键，绝对值大小不重要

### 4.4 争议与质疑

"Rethinking the Value of Network Pruning" 论文提出质疑：
- 彩票假说只在某些条件下成立（小学习率、非结构化剪枝）
- 学习率调大后观察不到彩票假说现象
- 彩票假说和这篇质疑论文**同时出现在ICLR 2019**

**结论：** 彩票假说仍然是一个有待更多研究证实的假说，不是定理。

In [ ]:
# 简化的彩票假说验证实验
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

class SimplePruning:
    """简单网络剪枝实现"""
    @staticmethod
    def prune_by_magnitude(model, prune_ratio=0.1):
        """按绝对值大小剪枝：绝对值最小的参数被置零"""
        all_weights = []
        for name, param in model.named_parameters():
            if 'weight' in name:
                all_weights.append(param.data.abs().flatten())
        all_weights = torch.cat(all_weights)
        threshold = torch.quantile(all_weights, prune_ratio)
        
        pruned_count = 0
        total_count = 0
        for name, param in model.named_parameters():
            if 'weight' in name:
                mask = (param.data.abs() > threshold).float()
                pruned_count += (mask == 0).sum().item()
                total_count += mask.numel()
                param.data = param.data * mask
        
        sparsity = pruned_count / total_count
        return sparsity
    
    @staticmethod
    def prune_neuron(layer, prune_ratio=0.1):
        """神经元剪枝：基于L2范数评估重要性"""
        neuron_norms = layer.weight.data.norm(dim=1)  # [out_features]
        threshold = torch.quantile(neuron_norms, prune_ratio)
        mask = (neuron_norms > threshold).float()
        
        layer.weight.data = layer.weight.data * mask.unsqueeze(1)
        if layer.bias is not None:
            layer.bias.data = layer.bias.data * mask
        
        sparsity = (mask == 0).sum().item() / mask.numel()
        return sparsity

print("剪枝工具函数定义完成")
print("迭代剪枝策略：每次剪10%，微调，再剪，重复")
print("神经元剪枝 > 权重剪枝（GPU友好）")

In [ ]:
# 彩票假说的核心实验逻辑
import torch
import torch.nn as nn
import copy

def lottery_ticket_experiment_logic():
    """
    彩票假说的三步实验逻辑：
    1. 随机初始化大网络 → 训练 → 剪枝 → 找到"中奖"子网络
    2. 子网络重新随机初始化 → 重训练 → 训不起来！
    3. 子网络用原大网络中对应的初始化参数 → 训起来了！
    """
    print("彩票假说实验验证：")
    print()
    print("步骤1：随机初始 → 训练大网络 → 剪枝找'中奖'子网")
    print("  θ_init = N(0,1)  # 随机初始化")
    print("  θ_trained = SGD(θ_init, data)  # 训练")
    print("  θ_winning = prune(θ_trained)   # 剪枝，保留重要的")
    print()
    print("步骤2：子网络重新随机初始化 → 训不起来")
    print("  θ_new_init = N(0,1)  # 新的随机初始化")
    print("  θ_fail = SGD(θ_new_init, data)  # 训不起来！")
    print()
    print("步骤3：用原初始化 → 训起来了")
    print("  θ_same_init = θ_init[mask]  # 复制原初始化中对应的参数")
    print("  θ_success = SGD(θ_same_init, data)  # 训起来了！")
    print()
    print("结论：不是结构好，是那组初始化参数'中奖'了")
    print()
    print("后续发现：正负号才是关键！")
    print("  θ = sign(θ_init) * α  # 保留正负号，统一幅度")
    print("  效果 ≈ 原始初始化！")

lottery_ticket_experiment_logic()

## 5. 知识蒸馏 (Knowledge Distillation) 详解

### 5.1 核心流程

1. 训练一个大的**教师网络**（准确率高但大/慢）
2. 训练一个小的**学生网络**，不直接学标签，而是学教师的输出分布
3. 学生网络 = 真正部署的模型

### 5.2 温度参数 T 的秘密

标准 Softmax（用于分类输出）：

$$y_i' = \frac{\exp(y_i)}{\sum_j \exp(y_j)}$$

带温度的 Softmax（用于知识蒸馏）：

$$y_i' = \frac{\exp(y_i/T)}{\sum_j \exp(y_j/T)}$$

**温度的作用：**

假设教师网络原始输出为 $y_1=100, y_2=10, y_3=1$：

| 温度 T | 类别1 | 类别2 | 类别3 | 效果 |
|------|------|------|------|------|
| T=1（标准） | 1.0 | ~0 | ~0 | 几乎One-Hot |
| T=100 | 0.56 | 0.23 | 0.21 | 分布平滑（有用！） |

**直觉：** 教师说"1是0.7分，7是0.2分"，这透露了"1和7有点像"的信息——这些类间关系在One-Hot标签中完全不存在！

### 5.3 不止最后一层

除了匹配教师网络的最终输出，还可以匹配中间层的输出：
- 学生网络第6层 ≈ 教师网络第12层
- 学生网络第3层 ≈ 教师网络第6层

这种多层匹配往往能获得更好的蒸馏效果。

### 5.4 集成蒸馏

比赛常用的技巧：训练1000个模型取平均（集成）→ 效果好但推理太慢
→ 用知识蒸馏将集成结果压缩为单个学生模型 → 既保持性能又只需一次推理

In [ ]:
# 知识蒸馏的完整 PyTorch 实现
import torch
import torch.nn as nn
import torch.nn.functional as F

class KnowledgeDistillationLoss(nn.Module):
    """
    知识蒸馏损失
    L = α * T² * KL(softmax(student/T) || softmax(teacher/T)) 
        + (1-α) * CE(student, true_labels)
    """
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
    
    def forward(self, student_logits, teacher_logits, true_labels):
        # 蒸馏损失（软标签）
        soft_student = F.log_softmax(student_logits / self.temperature, dim=1)
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=1)
        loss_distill = F.kl_div(soft_student, soft_teacher, reduction='batchmean')
        loss_distill *= self.temperature ** 2  # 梯度缩放
        
        # 硬标签损失
        loss_hard = F.cross_entropy(student_logits, true_labels)
        
        return self.alpha * loss_distill + (1 - self.alpha) * loss_hard

# 演示温度效应
print("温度 T 对 Softmax 分布的影响：")
print()
logits = torch.tensor([[5.0, 2.0, 0.5, 0.1, 0.05]])
for T in [1, 2, 4, 10, 100]:
    probs = F.softmax(logits / T, dim=1)
    print(f"T={T:3d}: {[f'{p:.4f}' for p in probs[0].tolist()]}")
print()
print("T=1:   几乎 One-Hot → 和直接用标签一样")
print("T=4:   适度平滑 → 揭示类间关系")
print("T=100: 近乎均匀 → 信息丢失")

## 6. 参数量化 (Parameter Quantization)

### 6.1 降低精度

| 精度 | 每个参数位数 | 说明 |
|------|------|------|
| FP32 | 32bit | 标准训练 |
| FP16 | 16bit | 混合精度训练 |
| INT8 | 8bit | 推理加速 |
| INT4 | 4bit | 进一步压缩 |
| Binary | 1bit | 只有 +1/-1 |

### 6.2 权重聚类 + 查表法

1. 对网络所有权重做聚类（如分4群）
2. 每群用一个中心值代表（如 -0.4, 0.1, 0.5, 0.9）
3. 每个权重只需存储它属于哪一群（2bit/参数）
4. 再叠加哈夫曼编码进一步压缩

### 6.3 二值网络 (Binary Network)

- 所有权重只有 +1 或 -1
- **反直觉的好处**：二进制网络有时比全精度网络表现更好！
- **原因：** 强正则化防止过拟合——限制了网络容量，反而在测试集上更好
- BinaryConnect、XNOR-Net 等经典方法

In [ ]:
# 深度可分离卷积的 PyTorch 实现
import torch
import torch.nn as nn

class DepthwiseSeparableConv(nn.Module):
    """
    深度可分离卷积 = 深度卷积 + 点卷积
    参数量 ≈ 标准卷积的 1/K²
    这是 MobileNet 的核心模块
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        # 深度卷积：每个通道独立做空间卷积
        self.depthwise = nn.Conv2d(
            in_channels, in_channels,  # 输入输出通道数相同！
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels  # groups=in_channels 就是深度卷积
        )
        # 点卷积 (1x1)：只做通道间混合
        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            stride=1,
            padding=0
        )
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

# 参数量对比
def compare_params():
    in_c, out_c, k = 256, 256, 3
    
    # 标准卷积参数量
    std_params = k * k * in_c * out_c
    
    # 深度可分离卷积参数量
    ds_params = k * k * in_c + in_c * out_c  # 深度 + 点
    
    ratio = ds_params / std_params
    print(f"标准卷积参数量：    {std_params:,}")
    print(f"深度可分离卷积参数量：{ds_params:,}")
    print(f"比例：{ratio:.4f} ≈ 1/{1/ratio:.1f}")
    print()
    print(f"公式：(K²I + IO) / (K²IO) = 1/O + 1/K² ≈ 1/K²")
    print(f"当 K=3 时 ≈ 1/9 的参数量！")

compare_params()

## 7. 网络架构设计：深度可分离卷积

### 7.1 标准卷积回顾

输入 $C_{in}$ 个通道，输出 $C_{out}$ 个通道，核大小 $K \times K$：

$$\text{参数量} = K \times K \times C_{in} \times C_{out}$$

### 7.2 深度可分离卷积

**步骤一：深度卷积 (Depthwise Convolution)**
- 每个通道有自己独立的 $K \times K$ 滤波器
- 输入 $C_{in}$ 通道 → 输出 $C_{in}$ 通道（通道数不变）
- 参数量：$K \times K \times C_{in}$

**步骤二：点卷积 (Pointwise Convolution)**
- $1 \times 1$ 卷积，只做通道间混合
- 输入 $C_{in}$ 通道 → 输出 $C_{out}$ 通道
- 参数量：$C_{in} \times C_{out}$

**总参数量对比：**

$$\frac{K^2 C_{in} + C_{in} C_{out}}{K^2 C_{in} C_{out}} = \frac{1}{C_{out}} + \frac{1}{K^2} \approx \frac{1}{K^2}$$

当 $K=3$ 时，约 **1/9** 的参数量！这是 MobileNet 系列成功的关键。

### 7.3 与低秩近似的关系

把一个标准卷积层拆成两层，等价于对权重矩阵做低秩分解：

$$W \approx U \times V$$

其中 $W$ 是 $M \times N$ 的大矩阵，拆成 $U$（$M \times K$）和 $V$（$K \times N$），只要 $K$ 够小就能大幅减少参数量。但这样做也限制了 $W$ 的秩（rank $\leq K$），不是所有参数矩阵都能被表示。

## 8. 动态计算 (Dynamic Computation)

### 8.1 为什么需要动态计算？

- 同一个模型可能在不同设备上运行（手机 vs 服务器）
- 同一设备上计算资源也会变化（电量充足 vs 低电量）
- 不同样本难度不同（简单图片可能一层就够了）

与其准备10个不同大小的模型，不如训练**一个能自适应调整计算量的模型**。

### 8.2 动态深度

在每层后都加一个分类器：

$$L = e_1 + e_2 + \cdots + e_L$$

同时最小化所有层的分类损失。简单样本可能第一层就能分类，困难样本需要跑完全部层。

### 8.3 动态宽度

同一网络可以选择不同的宽度（如100%/75%/50%的通道数）：

$$L = e_1 + e_2 + e_3$$

所有宽度的输出都要求与正确答案接近。参考 Slimmable Neural Networks。

### 8.4 让网络自己决定

- 网络根据输入难度自动决定在第几层停止
- 例如：SkipNet, BlockDrop, Runtime Neural Pruning

In [ ]:
# 深度可分离卷积参数量对比（公式演示）
print("=" * 60)
print("深度可分离卷积 vs 标准卷积：参数量对比")
print("=" * 60)
print()

for k in [2, 3, 5]:
    for c_in, c_out in [(64, 128), (256, 256), (512, 512)]:
        std = k * k * c_in * c_out
        ds = k * k * c_in + c_in * c_out
        ratio = ds / std
        print(f"K={k}, C_in={c_in:3d}, C_out={c_out:3d}: "
              f"标准={std:>10,d}  DS={ds:>8,d}  比例={ratio:.4f}")
    print()

print("结论：深度可分离卷积约节省 80-90% 的参数！")
print("实际精度损失通常 < 1%")

## 9. 常见误区与易错点

### 误区 1：剪枝后权重真正被删除了
**纠正：** 权重剪枝实践中通常将权重设零而非真正删除（因为不规则稀疏矩阵GPU加速困难）。真正有效的剪枝是神经元级别的。

### 误区 2：知识蒸馏只是让学生模仿教师
**纠正：** 蒸馏的精髓在于温度 T 揭示了类间关系——这些信息不存在于One-Hot标签中。

### 误区 3：量化一定损失精度
**纠正：** 适当的量化（如INT8）精度损失微乎其微，有时甚至因为正则化效果更好。

### 误区 4：彩票假说是定理
**纠正：** 彩票假说是一个假说（hypothesis），有支持的实验也有反对的实验。特定条件（小学习率、非结构化剪枝）下成立。

### 误区 5：网络压缩技术只能用一种
**纠正：** 五种技术是互补的，可以组合：架构设计 → 知识蒸馏 → 剪枝 → 量化。

### 误区 6：深度可分离卷积和低秩近似是两个不同技术
**纠正：** 深度可分离卷积本质上就是把一个标准卷积拆成两层——这等价于低秩近似。

## 10. 与其他章节的联系

| 章节 | 联系 |
|------|------|
| 第4章 CNN | 深度可分离卷积是标准卷积的变体 |
| 第10章 自监督学习 | BERT/GPT等大模型是网络压缩的主要对象 |
| 第15章 元学习 | NAS 是元学习在网络架构搜索中的应用 |
| 第19章 ChatGPT | GPT系列有上千亿参数，网络压缩是将其部署到端侧的关键 |

## 11. 核心公式汇总

### 标准 Softmax

$$y_i' = \frac{\exp(y_i)}{\sum_j \exp(y_j)}$$

### 带温度的 Softmax（知识蒸馏用）

$$y_i' = \frac{\exp(y_i/T)}{\sum_j \exp(y_j/T)}$$

### 深度学习可分卷积参数量比例

$$\frac{K^2 I + I O}{K^2 I O} = \frac{1}{O} + \frac{1}{K^2} \approx \frac{1}{K^2}$$

### 选择性的突触可塑性损失（终身学习相关）

$$L'(\theta) = L(\theta) + \lambda \sum_i b_i (\theta_i - \theta_i^{\text{old}})^2$$

### 动态深度总损失

$$L = e_1 + e_2 + \cdots + e_L$$

### 动态宽度总损失

$$L = e_1 + e_2 + e_3$$

## 12. 关键总结

1. **五种技术互补**：网络架构设计、知识蒸馏、剪枝、量化、动态计算可组合使用
2. **神经元剪枝优于权重剪枝**：因为网络保持规整，GPU友好
3. **彩票假说揭示了初始化的重要性**：大网络好训是因为包含"幸运"子网络
4. **初始化参数的正负号比数值更重要**：Deconstructing Lottery Tickets 的核心发现
5. **知识蒸馏的精髓在温度 T**：温度揭示类间关系，这是One-Hot标签不具备的
6. **深度可分离卷积 ≈ 1/K² 参数量**：MobileNet的核心思想，将卷积拆成空间+通道两步
7. **量化有时能提升性能**：对网络容量加限制 = 正则化效果
8. **动态计算让一个模型适配多种场景**：不需要准备多个不同大小的模型
9. **彩票假说非定理**：有支持和反对的证据，取决于具体实验条件
10. **压缩技术的实用价值**：让大模型能在手机、手表等边缘设备上运行